Goal:
- Build an AI assistant that can answer underwriting questions.
- Uses RAG: retrieves similar policies and generates explanations.
- Maintains interpretability and context for decisions.

In [16]:
import pandas as pd

df = pd.read_csv("insurance_dataset.csv")

In [17]:
import joblib

tweedie_model = joblib.load("tweedie_model.pkl")
tweedie_feature_columns = joblib.load("tweedie_feature_columns.pkl")

In [19]:
xgb_model = joblib.load("xgb_model.pkl")
xgb_feature_columns = joblib.load("xgb_feature_columns.pkl")

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   age               50000 non-null  int64  
 1   driver_type       50000 non-null  object 
 2   years_experience  50000 non-null  int64  
 3   income_band       50000 non-null  object 
 4   vehicle_age       50000 non-null  int64  
 5   vehicle_type      50000 non-null  object 
 6   vehicle_value     50000 non-null  float64
 7   airbags           50000 non-null  int64  
 8   tracking_device   50000 non-null  int64  
 9   region            50000 non-null  object 
 10  traffic_density   50000 non-null  object 
 11  policy_duration   50000 non-null  int64  
 12  annual_mileage    50000 non-null  int64  
 13  speeding_score    50000 non-null  object 
 14  previous_claims   50000 non-null  int64  
 15  fraud_flag        50000 non-null  int64  
 16  risk_score        50000 non-null  float6

### Preparing Knowledge Base (all policies, key features, model logic.) 

We’ll convert each policy into a text chunk that can be embedded/retrieved.

In [21]:
# Adding core features to the knowledge base
kb_features = ["age", "driver_type", "years_experience", "vehicle_type", "vehicle_value", "vehicle_age", "annual_mileage", "previous_claims", "risk_score", "airbags", "tracking_device", "region", "policy_duration"]

# Creating text representation of each policy
def policy_to_text(row):
    return (
        f"Policyholder: age {row['age']}, driver_type {row['driver_type']}, years_experience {row['years_experience']}, "
        f"vehicle_type {row['vehicle_type']}, vehicle_value {row['vehicle_value']}, ({row['vehicle_age']} yrs old), "
        f"annual_mileage {row['annual_mileage']}, previous_claims {row['previous_claims']}, "
        f"safety: airbags {row['airbags']}, tracking_device {row['tracking_device']}, "
        f"region {row['region']}, policy_duration {row['policy_duration']} months, "
        f"risk_score {row['risk_score']:.2f}."
    )

df["policy_text"] = df.apply(policy_to_text, axis=1)

In [22]:
df["policy_text"].head(5)

0    Policyholder: age 59, driver_type private, yea...
1    Policyholder: age 49, driver_type taxi, years_...
2    Policyholder: age 35, driver_type private, yea...
3    Policyholder: age 63, driver_type taxi, years_...
4    Policyholder: age 28, driver_type taxi, years_...
Name: policy_text, dtype: object

### Vectorizing Policies
Using TF-IDF embeddings for similarity (works well for tabular-to-text embeddings).

TF-IDF vectorization converts policy text into numerical feature vectors that capture how important each word is in describing risk patterns across policies (common words = less important, rare but meaningful words = more important).

- TF (Term Frequency) - how often a word appears in a policy
- IDF (Inverse Document Frequency) - how rare/important that word is across all policies

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000) # to avoid overfitting & too sparse vectors
policy_vectors = vectorizer.fit_transform(df["policy_text"])

In [24]:
policy_vectors.shape

(50000, 5000)

### Retrieval Function

In [25]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_similar_policies(query_text, top_k=5):
    query_vec = vectorizer.transform([query_text])
    sims = cosine_similarity(query_vec, policy_vectors).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return df.iloc[top_idx][["policy_text", "risk_score", "claim_occurred"]], sims[top_idx]

In [26]:
query = "Private driver, 28 yrs, sedan, 5 yrs old, 18000 km/yr, 0 previous claims, Nairobi, 12 months policy"
similar_policies, scores = retrieve_similar_policies(query)
similar_policies

,policy_text,risk_score,claim_occurred
37359,"Policyholder: age 28, driver_type private, yea...",0.28,0
3195,"Policyholder: age 28, driver_type private, yea...",0.28,0
4789,"Policyholder: age 30, driver_type private, yea...",0.28,0
45715,"Policyholder: age 46, driver_type private, yea...",0.28,0
10490,"Policyholder: age 46, driver_type private, yea...",0.28,0


### Groq Setup

We’ll combine retrieved policies and user question into a prompt

In [27]:
def predict_pure_premium(input_dict):
    input_df = pd.DataFrame([input_dict])
    
    input_df = pd.get_dummies(input_df)
    # Align with training columns
    input_df = input_df.reindex(columns=tweedie_feature_columns, fill_value=0)
    
    # Predict expected loss (pure premium)
    pure_premium = tweedie_model.predict(input_df)[0]
    
    return pure_premium

def calculate_final_premium(pure_premium, expense_loading=0.3, profit_margin=0.1):
    return pure_premium * (1 + expense_loading + profit_margin)

In [28]:
def calculate_premium(predicted_risk, vehicle_value,
                      expense_loading=0.3, profit_margin=0.1):    
    # Expected loss (pure premium)
    expected_loss = predicted_risk * vehicle_value
    
    # Adding loadings
    premium = expected_loss * (1 + expense_loading + profit_margin)
    premium = max(premium, 5000)  # avoid unrealistically low premiums
    
    return expected_loss, premium

In [29]:
def predict_risk_from_query(query_dict):
    input_df = pd.DataFrame([query_dict])
    
    # Converting categoricals
    input_df = pd.get_dummies(input_df)
    
    # Aligning with training features
    input_df = input_df.reindex(columns=xgb_feature_columns, fill_value=0)
    
    return xgb_model.predict_proba(input_df)[:, 1][0]

In [30]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = ChatGroq(
    temperature=0.3,
    groq_api_key=GROQ_API_KEY,
    model_name="llama-3.3-70b-versatile"
)

def ask_groq(prompt):
    response = client.invoke([
        {"role": "system", "content": "You are an expert insurance underwriter."},
        {"role": "user", "content": prompt}
    ])
    
    return response.content

In [31]:
def build_prompt(user_question, retrieved_policies, predicted_risk, pure_premium, premium):
    context = "\n".join(retrieved_policies["policy_text"].tolist())
    prompt = f"""
You are an AI insurance underwriting assistant.

RISK SCORE: {predicted_risk:.2f}
PURE PREMIUM (EXPECTED LOSS): {pure_premium:.0f}
RECOMMENDED PREMIUM: {premium:.0f}

SIMILAR POLICIES:
{context}

QUESTION:
{user_question}

INSTRUCTIONS:
- Provide underwriting decision (Approve / Conditional / Decline)
- Explain reasoning using risk score and similar cases
- Highlight key risk drivers
- Justify the recommended premium

ANSWER:
"""
    return prompt

### Testing the RAG Assistant

In [32]:
query_features = {
    "age": 30,
    "driver_type": "private",
    "vehicle_age": 3,
    "vehicle_type": "sedan",
    "vehicle_value": 1200000,
    "annual_mileage": 15000,
    "previous_claims": 0,
    "airbags": 4,
    "tracking_device": 1,
    "region": "Nairobi",
    "policy_duration": 12,
    "speeding_score": "medium"
}

In [33]:
import requests

# Example quiz
user_question = "What is the risk profile for a 30-year-old private driver with a 3-year-old sedan, valued at 1200000, annual mileage 15000, 4 airbags, 1 tracking device, in Nairobi, no previous claims?"

retrieved_policies, sims = retrieve_similar_policies(user_question)
predicted_risk = predict_risk_from_query(query_features)
pure_premium = predict_pure_premium(query_features)
premium = calculate_final_premium(pure_premium)
prompt = build_prompt(user_question, retrieved_policies, predicted_risk, pure_premium, premium)

answer = ask_groq(prompt)

print(answer)

**Underwriting Decision:** Approve

**Reasoning:**
The predicted risk score for this policyholder is 0.11, which is lower than the risk scores of similar policies (0.30). This suggests that the policyholder has a lower risk profile compared to the similar cases. The similar policies have a higher risk score, which may be due to factors such as older vehicles, higher annual mileage, or longer policy durations.

The policyholder's profile is similar to the first two similar policies, with a few key differences. The policyholder is 30 years old, with a 3-year-old sedan, valued at 1200000, which is comparable to the first two similar policies. The annual mileage of 15000 is also similar to the first two similar policies. The presence of 4 airbags and 1 tracking device suggests a good safety profile.

**Justification of Recommended Premium:**
The pure premium (expected loss) is 29714, and the recommended premium is 41599. The recommended premium is higher than the pure premium, which sugges

In [34]:
vehicle_value = query_features["vehicle_value"]

expected_loss, recommended_premium = calculate_premium(predicted_risk, vehicle_value)

print(f"Expected Loss: {expected_loss:.2f}")
print(f"Recommended Premium: {recommended_premium:.2f}")

Expected Loss: 130977.70
Recommended Premium: 183368.77


In [35]:
risk_avg = retrieved_policies["risk_score"].mean()
claim_rate = retrieved_policies["claim_occurred"].mean()

print(f"Average risk_score of similar policies: {risk_avg:.2f}")
print(f"Claim rate among similar policies: {claim_rate:.2%}")

Average risk_score of similar policies: 0.30
Claim rate among similar policies: 0.00%
